In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('../raw/epa_ghgrp_2021_2023_aggregate.csv')

# Initial exploration
print("Dataset Overview -----------------")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

Dataset Overview -----------------
Shape: (19513, 7)

Columns: ['facility_name', 'state', 'industry_sector', 'total_ghg_emissions_tonnes', 'latitude', 'longitude', 'reporting_year']

Data types:
facility_name                  object
state                          object
industry_sector                object
total_ghg_emissions_tonnes    float64
latitude                      float64
longitude                     float64
reporting_year                  int64
dtype: object

First few rows:


,facility_name,state,industry_sector,total_ghg_emissions_tonnes,latitude,longitude,reporting_year
0,121 REGIONAL DISPOSAL FACILITY,TX,Waste,314493.750,33.298570,-96.535860,2021
1,15-18565/15-18662,KY,Other,112348.750,37.274127,-83.239034,2021
2,1500 South Tibbs LLC d/b/a Aurorium Indianapol...,IN,Chemicals,64820.056,39.730000,-86.260000,2021
3,23rd and 3rd,NY,Power Plants,46081.780,40.663000,-74.000000,2021
4,31st Street Landfill,IL,Waste,7750.748,41.834962,-87.916392,2021


In [3]:
# Data Quality Analysis
print("Data Quality Analysis -----------------")
print(f"Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Basic statistics
print(f"\nBasic Statistics -----------------")
print(df.describe())

# Unique values in categorical columns
print(f"\nCategorical Data Analysis -----------------")
print(f"Unique states: {df['state'].nunique()}")
print(f"Unique industry sectors: {df['industry_sector'].nunique()}")
print(f"Unique reporting years: {sorted(df['reporting_year'].unique())}")

print(f"\nTop 10 industry sectors by count:")
print(df['industry_sector'].value_counts().head(10))

Data Quality Analysis -----------------
Missing values per column:
facility_name                 0
state                         0
industry_sector               0
total_ghg_emissions_tonnes    0
latitude                      0
longitude                     0
reporting_year                0
dtype: int64

Duplicate rows: 0

Basic Statistics -----------------
       total_ghg_emissions_tonnes      latitude     longitude  reporting_year
count                1.951300e+04  19513.000000  19513.000000    19513.000000
mean                 3.797793e+05     37.256437    -92.467454     2021.996976
std                  1.101691e+06      5.982582     16.878623        0.816208
min                  0.000000e+00     13.297100   -174.113611     2021.000000
25%                  3.243143e+04     32.789533    -97.856833     2021.000000
50%                  6.547875e+04     37.742800    -90.265666     2022.000000
75%                  1.853174e+05     41.219670    -82.635408     2023.000000
max              

In [4]:
# Outlier Analysis for GHG Emissions
print("Outlier Analysis -----------------")
Q1 = df['total_ghg_emissions_tonnes'].quantile(0.25)
Q3 = df['total_ghg_emissions_tonnes'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['total_ghg_emissions_tonnes'] < lower_bound) | 
              (df['total_ghg_emissions_tonnes'] > upper_bound)]
print(f"Number of outliers: {len(outliers)}")
print(f"Percentage of outliers: {len(outliers)/len(df)*100:.2f}%")

print(f"\nEmissions Statistics:")
print(f"Min: {df['total_ghg_emissions_tonnes'].min():,.2f}")
print(f"Max: {df['total_ghg_emissions_tonnes'].max():,.2f}")
print(f"Mean: {df['total_ghg_emissions_tonnes'].mean():,.2f}")
print(f"Median: {df['total_ghg_emissions_tonnes'].median():,.2f}")

# Visualize distribution
fig = make_subplots(rows=2, cols=2, 
                    subplot_titles=('Emissions Distribution', 'Log-scale Distribution', 
                                   'Box Plot', 'Emissions by Year'))

# Histogram
fig.add_trace(go.Histogram(x=df['total_ghg_emissions_tonnes'], name='Emissions'),
              row=1, col=1)

# Log-scale histogram
fig.add_trace(go.Histogram(x=np.log10(df['total_ghg_emissions_tonnes']), name='Log Emissions'),
              row=1, col=2)

# Box plot
fig.add_trace(go.Box(y=df['total_ghg_emissions_tonnes'], name='Emissions'),
              row=2, col=1)

# Emissions by year
yearly_stats = df.groupby('reporting_year')['total_ghg_emissions_tonnes'].agg(['mean', 'median', 'sum']).reset_index()
fig.add_trace(go.Scatter(x=yearly_stats['reporting_year'], y=yearly_stats['sum'], 
                        mode='lines+markers', name='Total Emissions'),
              row=2, col=2)

fig.update_layout(height=800, title_text="GHG Emissions Analysis", showlegend=False)
fig.show()

Outlier Analysis -----------------
Number of outliers: 3128
Percentage of outliers: 16.03%

Emissions Statistics:
Min: 0.00
Max: 21,775,439.59
Mean: 379,779.26
Median: 65,478.75


/Users/sandratang/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

divide by zero encountered in log10



In [ ]:
# Top Emitting Sectors Analysis
print("Top Emitting Sectors Analysis -----------------")
sector_stats = df.groupby('industry_sector')['total_ghg_emissions_tonnes'].agg(['count', 'sum', 'mean']).reset_index()
sector_stats = sector_stats.sort_values('sum', ascending=False)

print("Top 10 sectors by total emissions:")
print(sector_stats.head(10))

# Visualize top 10 emitting sectors
fig = px.bar(sector_stats.head(10), 
             x='sum', y='industry_sector',
             title='Top 10 Industry Sectors by Total GHG Emissions (2021-2023)',
             labels={'sum': 'Total GHG Emissions (tonnes)', 'industry_sector': 'Industry Sector'},
             orientation='h')
fig.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
fig.show()

# Emissions by State
state_stats = df.groupby('state')['total_ghg_emissions_tonnes'].agg(['count', 'sum', 'mean']).reset_index()
state_stats = state_stats.sort_values('sum', ascending=False)

print(f"\nTop 10 states by total emissions:")
print(state_stats.head(10))

# Visualize emissions by state
fig = px.bar(state_stats.head(15), 
             x='state', y='sum',
             title='Top 15 States by Total GHG Emissions (2021-2023)',
             labels={'sum': 'Total GHG Emissions (tonnes)', 'state': 'State'})
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [ ]:
# Yearly Trends Analysis
print("Yearly Trends Analysis (2021-2023) -----------------")
yearly_trends = df.groupby('reporting_year').agg({
    'total_ghg_emissions_tonnes': ['count', 'sum', 'mean'],
    'facility_name': 'nunique'
}).round(2)

yearly_trends.columns = ['Facility_Count', 'Total_Emissions', 'Mean_Emissions', 'Unique_Facilities']
yearly_trends = yearly_trends.reset_index()
print(yearly_trends)

# Calculate year-over-year changes
yearly_trends['Total_Change_Pct'] = yearly_trends['Total_Emissions'].pct_change() * 100
yearly_trends['Mean_Change_Pct'] = yearly_trends['Mean_Emissions'].pct_change() * 100

print(f"\nYear-over-Year Changes:")
print(yearly_trends[['reporting_year', 'Total_Change_Pct', 'Mean_Change_Pct']])

# Visualize yearly trends
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Total Emissions by Year', 'Average Emissions by Year',
                                   'Number of Facilities by Year', 'Top Sectors Trend'))

# Total emissions
fig.add_trace(go.Scatter(x=yearly_trends['reporting_year'], y=yearly_trends['Total_Emissions'],
                        mode='lines+markers', name='Total Emissions', line=dict(width=3)),
              row=1, col=1)

# Average emissions
fig.add_trace(go.Scatter(x=yearly_trends['reporting_year'], y=yearly_trends['Mean_Emissions'],
                        mode='lines+markers', name='Mean Emissions', line=dict(width=3)),
              row=1, col=2)

# Number of facilities
fig.add_trace(go.Scatter(x=yearly_trends['reporting_year'], y=yearly_trends['Facility_Count'],
                        mode='lines+markers', name='Facility Count', line=dict(width=3)),
              row=2, col=1)

# Top sectors over time
top_sectors = df['industry_sector'].value_counts().head(5).index
for sector in top_sectors:
    sector_yearly = df[df['industry_sector'] == sector].groupby('reporting_year')['total_ghg_emissions_tonnes'].sum()
    fig.add_trace(go.Scatter(x=sector_yearly.index, y=sector_yearly.values,
                            mode='lines+markers', name=sector[:15]),
                  row=2, col=2)

fig.update_layout(height=800, title_text="GHG Emissions Trends Over Time", showlegend=True)
fig.show()